In [ ]:
import pickle
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))
from operators import simulate

with open("../data/problem_data_clean.pkl", "rb") as f:
     problem_data = pickle.load(f)
with open("../data/problem_data_clean_less_receivers.pkl", "rb") as f:
     problem_data_less_receivers = pickle.load(f)

receiver_data = problem_data["receiver_data"] # (Time x Receivers)
receiver_coords = problem_data["receiver_coords"] # (Receivers x 2) [x, z]
dt = problem_data["dt"] # Sampling interval

receiver_data2 = problem_data_less_receivers["receiver_data"] # (Time x Receivers)
receiver_coords2 = problem_data_less_receivers["receiver_coords"] # (Receivers x 2) [x, z]
dt2 = problem_data_less_receivers["dt"] # Sampling interval



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

receiver_x = receiver_coords[:, 0]
receiver_z = receiver_coords[:, 1]
x_padding = max(np.ptp(receiver_x) * 0.02, 1e-3)
max_depth = max(float(receiver_z.max()) * 1.1, float(receiver_z.max()) + 0.01)

fig, ax = plt.subplots(figsize=(20, 4))
ax.scatter(
    receiver_x,
    receiver_z,
    marker="^",
    s=5,
    label="Receivers",
    edgecolors="none",
    linewidths=0,
)
ax.set_title("Receiver Locations on Grid")
ax.set_xlabel("Lateral Dimension, x (m)")
ax.set_ylabel("Depth, z (m)")
ax.set_xlim(receiver_x.min() - x_padding, receiver_x.max() + x_padding)
ax.set_ylim(max_depth, 0)
ax.legend(markerscale=3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

time_us = np.arange(receiver_data.shape[0]) * dt * 1e6
num_traces = min(20, receiver_data.shape[1])
traces = receiver_data[:, :num_traces]
max_amplitude = np.max(np.abs(traces))
trace_scale = 0.45 / max_amplitude if max_amplitude else 0

fig, (ax, trace_ax) = plt.subplots(
    nrows=2,
    sharex=True,
    figsize=(12, 16),
    gridspec_kw={"height_ratios": [3, 3]},
)
ax.imshow(
    receiver_data.T,
    aspect="auto",
    extent=[time_us[0], time_us[-1], receiver_data.shape[1], 0],
    vmin=-100, vmax=100,
)
ax.set_title("Receiver Data")
ax.set_ylabel("Receiver Channel")

for channel in range(num_traces):
    trace_ax.plot(
        time_us,
        channel + traces[:, channel] * trace_scale,
        linewidth=1,
    )
trace_ax.set_title(f"First {num_traces} Receiver Traces")
trace_ax.set_xlabel("Time (us)")
trace_ax.set_ylabel("Receiver Channel (trace offset)")
trace_ax.set_xlim(time_us[0], time_us[-1])
trace_ax.set_ylim(num_traces - 0.5, -0.5)
trace_ax.set_yticks(range(num_traces))

fig.tight_layout()
plt.show()